In [2]:
# ============================================================
# Finale Klassifikation Level II — Individual / CoEco / Landscape
# Modell: MobileCLIP-S1 + Mixed Prompts (Macro-F1 = 0.787)
# Input: non-human Bilder aus Level I
# ============================================================
import os
import shutil
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import mobileclip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

# --- Modell laden ---
weights_path = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"
model, _, preprocess = mobileclip.create_model_and_transforms(
    'mobileclip_s1',
    pretrained=weights_path,
    device=device
)
model.eval()
tokenizer = mobileclip.get_tokenizer('mobileclip_s1')
print("Modell geladen: MobileCLIP-S1")


# ============================================================
# Optimierte Mixed-Prompts (bestes Set aus Dev-Evaluation)
# MobileCLIP-S1 + Mixed: Macro-F1 = 0.787
# ============================================================
categories = ["individual", "community_ecosystem", "landscape"]

mixed_prompts = {
    "individual":          "a close-up photo with a single animal, plant, or object as the "
                           "clear main subject",
    "community_ecosystem": "a photo dominated by multiple organisms, people, buildings, "
                           "vehicles, or man-made structures",
    "landscape":           "a wide natural landscape photo with visible horizon or open "
                           "terrain, no dominant single subject or man-made structures"
}

category_texts = [mixed_prompts[cat] for cat in categories]
print("Prompts geladen: Mixed (MobileCLIP-S1 optimal)")


# ============================================================
# Klassifikationsfunktion
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None

BATCH_SIZE       = 32
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def classify(image_paths, category_texts):
    text_tokens = tokenizer(category_texts).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    results, confidences, valid_paths = [], [], []

    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Batches"):
        batch_paths = image_paths[i : i + BATCH_SIZE]
        loaded = [(p, safe_open(p)) for p in batch_paths]
        loaded = [(p, img) for p, img in loaded if img is not None]
        if not loaded:
            continue

        paths_ok, imgs_ok = zip(*loaded)
        img_tensors = torch.stack([preprocess(img) for img in imgs_ok]).to(device)

        with torch.no_grad():
            image_features = model.encode_image(img_tensors)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarities = image_features @ text_features.T
        batch_preds  = similarities.argmax(dim=1).cpu().numpy()
        batch_conf   = similarities.max(dim=1).values.cpu().float().numpy()

        results.extend([categories[idx] for idx in batch_preds])
        confidences.extend(batch_conf)
        valid_paths.extend(paths_ok)

    return valid_paths, results, confidences


# ============================================================
# Finale Klassifikation — non-human Bilder aus Level I
# ============================================================
full_dataset_folder = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Human_non_Human\final_class_human_non_human_LINEARPROBE_vitl14_sorted\non_human"
output_csv          = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape.csv"
output_base         = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_sorted"

full_image_paths = []
for root, _, files in os.walk(full_dataset_folder):
    for fname in sorted(files):
        ext = os.path.splitext(fname)[1].lower()
        if ext in VALID_EXTENSIONS:
            full_image_paths.append(os.path.join(root, fname))

print(f"Bilder gefunden: {len(full_image_paths)}")

valid_paths, results, confidences = classify(full_image_paths, category_texts)

# --- CSV speichern ---
df_results = pd.DataFrame({
    "file_name":  [os.path.basename(p) for p in valid_paths],
    "file_path":  valid_paths,
    "pred_label": results,
    "confidence": confidences
})
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df_results.to_csv(output_csv, index=False)

print(f"\n✅ Klassifikation abgeschlossen.")
print(f"Gespeichert: {output_csv}")
print("\nVerteilung im Gesamtdatensatz:")
print(df_results["pred_label"].value_counts())
print(f"\nGesamt klassifiziert: {len(df_results)} Bilder")


# ============================================================
# Bilder nach Vorhersage in Ordner sortieren
# ============================================================
sorted_dirs = {cat: os.path.join(output_base, cat) for cat in categories}
for d in sorted_dirs.values():
    os.makedirs(d, exist_ok=True)

for path, pred in tqdm(zip(valid_paths, results), total=len(valid_paths), desc="Kopiere Bilder"):
    dest = os.path.join(sorted_dirs[pred], os.path.basename(path))
    if not os.path.exists(dest):
        shutil.copy(path, dest)

print("\nBilder sortiert:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)} → {sorted_dirs[cat]}")

Gerät: cuda
Modell geladen: MobileCLIP-S1
Prompts geladen: Mixed (MobileCLIP-S1 optimal)
Bilder gefunden: 5240


Batches: 100%|███████████████████████████████████████████████████████████████████████| 164/164 [06:12<00:00,  2.27s/it]



✅ Klassifikation abgeschlossen.
Gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape.csv

Verteilung im Gesamtdatensatz:
pred_label
landscape              3686
community_ecosystem     992
individual              562
Name: count, dtype: int64

Gesamt klassifiziert: 5240 Bilder


Kopiere Bilder: 100%|█████████████████████████████████████████████████████████████| 5240/5240 [00:09<00:00, 571.23it/s]


Bilder sortiert:
  individual: 562 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_sorted\individual
  community_ecosystem: 992 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_sorted\community_ecosystem
  landscape: 3686 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_sorted\landscape
